# MCAST-ZINB + adj_visual
## 강남구 불법주정차 예측: SVI 시각 유사도 그래프 추가

**기존 모델(DCAST_ZINB_0412)과 달라진 점**
- `[SECTION 2]` SVI 파이프라인 전체 추가 (Map Matching → Gemma 4 캡션 → SBERT 임베딩 → adj_visual)
- `[SECTION 3]` adj_adapt: top-20 → **임계치 기반** 희소화로 변경
- `[SECTION 5]` STBlock: gcn_visual 추가, GCN 단순합 → **학습 가능한 가중합** (α·phys + β·adapt + γ·visual)
- `[SECTION 6]` 모든 학습/평가 함수에 adj_visual 파라미터 추가

## SECTION 0 : 설정 및 라이브러리

In [ ]:
# ── 0-1. Google Drive 마운트 ──────────────────────────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# ── 0-2. 패키지 설치 ──────────────────────────────────────────────────────────
!pip install geopandas shapely pyproj fiona rtree torch torchvision --quiet
!pip install sentence-transformers google-genai --quiet

In [ ]:
# ── 0-3. 라이브러리 임포트 ────────────────────────────────────────────────────
import os
import re
import glob
import json
import time
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import cKDTree
from shapely.geometry import Point

from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types

warnings.filterwarnings('ignore')
print("NumPy:", np.__version__, "| PyTorch:", torch.__version__,
      "| CUDA:", torch.cuda.is_available())

In [ ]:
# ── 0-4. API 키 설정 ─────────────────────────────────────────────────────────
# Colab 왼쪽 패널 → 🔑 Secrets → GOOGLE_API_KEY 추가 후 아래 실행
os.environ['GOOGLE_API_KEY_EH'] = userdata.get('GOOGLE_API_KEY_EH')
print("API 키 설정 완료" if os.environ.get('GOOGLE_API_KEY_EH') else "❌ API 키 없음")

In [ ]:
# ⚠️ 실행 전 이 셀의 경로를 본인 환경에 맞게 수정하세요 (데이터는 저장소에 미포함)
# ── 0-5. 경로 설정 (환경에 맞게 수정) ───────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/mcast_data'

ROAD_SHP     = f'{DRIVE_BASE}/도로 데이터/gangnam_road_connected_linkseg_4326_edited4-5_Filtered_2.shp'
PARKING_CSV  = f'{DRIVE_BASE}/불법주정차 데이터/강남구불법주정차중복제거_filtered.csv'
LANDUSE_SHP  = f'{DRIVE_BASE}/용도지역/UPIS_C_UQ111.shp'
POI_CSV      = f'{DRIVE_BASE}/poi_counts_per_link (1).csv'

SVI_JSON_DIR = f'{DRIVE_BASE}/거리영상_50m/json (1)'    # SVI 메타데이터 JSON 폴더
SVI_IMG_DIR  = f'{DRIVE_BASE}/거리영상_50m/images (1)'  # SVI 이미지 폴더

# 중간 결과 저장 경로
CAPTION_OUTPUT    = 'svi_captions.json'
EMBEDDING_OUTPUT  = 'svi_embeddings.npy'
SIM_MATRIX_OUTPUT = 'sim_matrix.npy'
ADJ_VISUAL_OUTPUT = 'adj_visual.pt'
ADJ_ADAPT_OUTPUT  = 'adj_adapt.pt'
KEYWORD_OUTPUT    = 'svi_keywords_by_link.json'   # ← 추가

In [ ]:
# ── 0-6. 하이퍼파라미터 ──────────────────────────────────────────────────────
# Gemma 4
GOOGLE_API_KEY  = os.environ.get('GOOGLE_API_KEY_EH', '')
GEMMA_MODEL     = 'gemma-4-31b-it'   # 또는 'gemma-4-26b-a4b-it' (MoE, 더 빠름)
GEMMA_RPM_LIMIT = 15                 # 무료 티어 기준 (유료: 60으로 조정)

# SBERT
SBERT_MODEL = 'all-mpnet-base-v2'   # 영어 유사도 태스크 최적 모델

# 임계치 — analyze_similarity_distribution() 실행 후 분포 보고 설정
ADAPT_THRESHOLD  = 0.8  # adj_adapt 임계치 (placeholder)
VISUAL_THRESHOLD = 0.8  # adj_visual 임계치 (placeholder)

# 모델
HIDDEN_DIM = 64
N_LAYERS   = 4
BATCH_SIZE = 32
EPOCHS     = 70
LR         = 5e-4
H_HORIZON = 24   # 예측 지평: 한 번에 미래 몇 시간을 예측할지

# 데이터 분할
TRAIN_START = '2022-07-01'; TRAIN_END = '2023-09-30'
VAL_START   = '2023-10-01'; VAL_END   = '2023-12-31'
TEST_START  = '2024-01-01'; TEST_END  = '2024-02-29'

# 시계열 윈도우
H_RECENT = 24; H_DAILY = 24; H_WEEKLY = 24

In [ ]:
# ── 0-7. 시드 고정 ────────────────────────────────────────────────────────────
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

---
## SECTION 1 : 전처리
기존 코드와 동일. 도로 버퍼 생성, 용도지역 조인, Map Matching, 시계열 pivot 생성.

In [ ]:
# ── 1-1. 도로 버퍼 생성 ───────────────────────────────────────────────────────
def assign_buffer_size_by_width(gdf, width_col='ROAD_BT', output_col='buffer_size'):
    """도로폭(m)에 따라 버퍼 크기 부여."""
    cond_list   = [gdf[width_col].isna() | (gdf[width_col] < 10),
                   gdf[width_col] < 15, gdf[width_col] < 40, gdf[width_col] >= 40]
    choice_list = [15, 25, 30, 50]
    gdf[output_col] = np.select(cond_list, choice_list)
    return gdf


def build_buffered_roads(road_gdf):
    """도로 GeoDataFrame → 버퍼 생성 + LINK_ID 부여."""
    assert road_gdf.crs.is_projected, '도로 데이터는 투영좌표계(EPSG:5179)여야 합니다.'
    road_gdf = road_gdf.copy()
    road_gdf['geometry_line']   = road_gdf.geometry
    road_gdf = assign_buffer_size_by_width(road_gdf)
    road_gdf['geometry_buffer'] = road_gdf.geometry.buffer(road_gdf['buffer_size'])
    buffered_roads = road_gdf.set_geometry('geometry_buffer').copy()
    buffered_roads['LINK_ID'] = buffered_roads.index.astype(int)
    return buffered_roads

In [ ]:
# ── 1-2. 용도지역 조인 ────────────────────────────────────────────────────────
def join_landuse_by_largest_overlap(buffered_roads, landuse_gdf):
    """도로 버퍼와 가장 큰 면적으로 겹치는 용도지역(MLSFC_CL) 조인."""
    overlaps = gpd.overlay(buffered_roads, landuse_gdf, how='intersection')
    overlaps['area'] = overlaps.geometry.area
    idx = overlaps.groupby('LINK_ID')['area'].idxmax()
    largest_overlap = overlaps.loc[idx, ['LINK_ID', 'MLSFC_CL']]
    buffered_roads  = buffered_roads.merge(largest_overlap, on='LINK_ID', how='left')
    buffered_roads['ROA_CLS_SE'] = pd.to_numeric(
        buffered_roads['ROA_CLS_SE'], errors='coerce').astype('Int64')
    buffered_roads['MLSFC_CL'] = (buffered_roads['MLSFC_CL']
        .astype(str).str.extract(r'(\d+)').astype('Int64'))
    return buffered_roads

In [ ]:
# ── 1-3. 포인트 → 도로 Map Matching (불법주정차 & SVI 공통 사용) ─────────────
def spatial_join_points_to_roads(df, buffered_roads,
                                  lon_col='경도', lat_col='위도', time_col='단속일시'):
    """
    위경도 포인트를 도로 버퍼에 공간 조인 → 가장 가까운 도로 선에 귀속.
    불법주정차 데이터와 SVI 데이터 모두 이 함수로 처리.
    """
    df = df.copy()
    df['geometry'] = gpd.points_from_xy(df[lon_col], df[lat_col])
    if time_col and time_col in df.columns:
        df['datetime'] = pd.to_datetime(df[time_col])

    point_gdf = (gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
                 .to_crs(epsg=5179)
                 .reset_index()
                 .rename(columns={'index': 'point_id'}))

    joined = gpd.sjoin(point_gdf,
                       buffered_roads[['LINK_ID', 'geometry_buffer']],
                       how='inner', predicate='intersects')

    line_ref = buffered_roads[['LINK_ID', 'geometry_line']].drop_duplicates('LINK_ID')
    joined   = joined.merge(line_ref, on='LINK_ID', how='left')
    joined['distance_to_line'] = joined.geometry.distance(joined['geometry_line'])

    closest_only = (joined
                    .loc[joined.groupby('point_id')['distance_to_line'].idxmin()]
                    .copy())
    drop_cols = ['index_right', 'geometry_line', 'geometry_buffer']
    closest_only.drop(columns=[c for c in drop_cols if c in closest_only.columns], inplace=True)

    matched, total = len(closest_only), len(df)
    print(f'✅ Map Matching 완료: {matched:,}/{total:,} ({matched/total*100:.1f}%)')
    return closest_only

In [ ]:
# ── 1-4. 시계열 pivot 생성 ────────────────────────────────────────────────────
def build_timeseries_pivot(closest_only, target_start_date, target_end_date,
                           all_road_link_ids):
    """LINK_ID × 시간(1시간 단위) count pivot 생성."""
    data_start = pd.to_datetime(target_start_date) - pd.Timedelta(hours=169)
    data_end   = pd.to_datetime(target_end_date)

    closest_only = closest_only.copy()
    closest_only['hour'] = closest_only['datetime'].dt.floor('h')
    mask = (closest_only['hour'] >= data_start) & (closest_only['hour'] <= data_end)

    timeseries = (closest_only[mask]
                  .groupby(['LINK_ID', 'hour'])
                  .size().reset_index(name='count'))
    pivot_df = timeseries.pivot(index='LINK_ID', columns='hour', values='count').fillna(0)

    full_time_range = pd.date_range(start=data_start, end=data_end, freq='1h')
    pivot_df = (pivot_df
                .reindex(columns=full_time_range, fill_value=0)
                .reindex(index=all_road_link_ids, fill_value=0))
    return pivot_df.sort_index(axis=0).sort_index(axis=1)


# ── 1-5. POI 병합 ─────────────────────────────────────────────────────────────
def merge_poi_counts(buffered_roads, poi_counts):
    poi_counts = poi_counts.copy()
    poi_counts['LINK_ID'] = poi_counts['LINK_ID'].astype('int64')
    return pd.merge(buffered_roads, poi_counts, on='LINK_ID', how='left')

In [ ]:
# ── 1-6. 모델 입력 텐서 생성 (Static + Temporal) ───────────────────────────────
def build_model_inputs(pivot_df, buffered_roads):
    """
    Static context (노드 특성)와 Temporal context (시간 임베딩) 생성.
    POI 그룹: 식음료/생활, 교육/보육, 교통/인프라, 의료/복지, 문화/여가
    """
    all_link_ids = pivot_df.index
    static_df = buffered_roads.copy()

    poi_groups = {
        'poi_comm'   : ['MT1', 'CS2', 'FD6', 'CE7'],
        'poi_edu'    : ['PS3', 'SC4', 'AC5'],
        'poi_trans'  : ['PK6', 'OL7', 'SW8'],
        'poi_pub'    : ['HP8', 'PM9', 'BK9', 'PO3', 'AG2'],
        'poi_leisure': ['CT1', 'AT4', 'AD5'],
    }
    for group_name, cats in poi_groups.items():
        existing = [c for c in cats if c in static_df.columns]
        static_df[group_name] = static_df[existing].sum(axis=1) if existing else 0

    target_cols = (['LINK_ID', 'ROAD_BT', 'ROA_CLS_SE', 'MLSFC_CL']
                   + list(poi_groups.keys()))
    static_df = (static_df[target_cols]
                 .set_index('LINK_ID').reindex(all_link_ids).fillna(0))
    static_df = pd.get_dummies(static_df, columns=['ROA_CLS_SE', 'MLSFC_CL'], dtype=float)

    scaler   = MinMaxScaler()
    num_cols = ['ROAD_BT', 'poi_comm', 'poi_edu', 'poi_trans', 'poi_pub', 'poi_leisure']
    static_df[num_cols] = scaler.fit_transform(static_df[num_cols])
    static_context = static_df.values

    time_index  = pivot_df.columns
    temporal_df = pd.DataFrame(index=time_index)
    temporal_df['hour_sin'] = np.sin(2 * np.pi * temporal_df.index.hour / 24)
    temporal_df['hour_cos'] = np.cos(2 * np.pi * temporal_df.index.hour / 24)
    temporal_df['day_sin']  = np.sin(2 * np.pi * temporal_df.index.dayofweek / 7)
    temporal_df['day_cos']  = np.cos(2 * np.pi * temporal_df.index.dayofweek / 7)
    temporal_context = temporal_df.values

    print(f'✅ Static {static_context.shape} | Temporal {temporal_context.shape}')
    return static_context, temporal_context, scaler

---
## SECTION 2 : SVI 파이프라인 (신규)
`SVI 메타데이터 로드 → Map Matching → Gemma 4 캡션 생성 → SBERT 임베딩 → Point-level Similarity → adj_visual`

In [ ]:
def load_svi_metadata(json_dir: str, image_dir: str) -> pd.DataFrame:
    """
    SVI JSON 메타데이터를 로드하여 이미지 경로와 함께 DataFrame 반환.
    JSON 구조:
        {
            "roadview": {
                "id"       : "loc_00001",
                "imgDate"  : "2023-05-10",
                "road_posX": 127.xxx,
                "road_posY": 37.xxx
            }
        }
    이미지 파일명 패턴: {location_id}_{direction}_1200_{date}.jpg
    """
    records    = []
    json_files = list(Path(json_dir).glob('*.json'))
    if not json_files:
        raise FileNotFoundError(f'JSON 파일 없음: {json_dir}')

    # ── 이미지 목록 1회 전체 스캔 후 딕셔너리 캐싱 (Drive I/O 최소화)
    print('📂 이미지 목록 캐싱 중...')
    img_cache = {}  # stem → 전체 경로
    for img_path in Path(image_dir).glob('*.jpg'):
        img_cache[img_path.stem] = str(img_path)
    print(f'   총 {len(img_cache):,}개 이미지 캐시 완료')

    for jf in json_files:
        with open(jf, 'r', encoding='utf-8') as f:
            data = json.load(f).get('roadview', {})

        loc_id  = data.get('id', jf.stem)
        date    = data.get('imgDate', '')
        svi_lon = data.get('road_posX', None)
        svi_lat = data.get('road_posY', None)

        if svi_lon is None or svi_lat is None:
            continue

        # ── glob 대신 캐시 딕셔너리 prefix 조회 (O(1) × 4)
        img_paths = {'back': None, 'front': None, 'left': None, 'right': None}
        for direction in ['back', 'front', 'left', 'right']:
            prefix  = f'{loc_id}_{direction}_1200_'
            matched = next((v for k, v in img_cache.items()
                            if k.startswith(prefix)), None)
            img_paths[direction] = matched

        records.append({'location_id': loc_id, 'date': date,
                        'svi_lon': svi_lon, 'svi_lat': svi_lat,
                        **img_paths})

    # ── DataFrame 생성
    df = pd.DataFrame(records)

    # ── 방향 컬럼 누락 방어 (레코드가 0개이거나 이미지가 전혀 없을 때)
    for col in ['back', 'front', 'left', 'right']:
        if col not in df.columns:
            df[col] = None

    complete = df.dropna(subset=['back', 'front', 'left', 'right'])
    print(f'✅ SVI 메타데이터: {len(complete)}/{len(df)} 지점 (4방향 완성)')

    if len(complete) == 0:
        print('⚠️  4방향 이미지 완성 지점 없음 — 아래 정보를 확인하세요:')
        print(f'   방향별 이미지 매칭 수:\n{df[["back","front","left","right"]].notna().sum()}')
        print(f'   loc_id 샘플: {df["location_id"].head(3).tolist()}')
        print(f'   이미지 캐시 샘플: {list(img_cache.keys())[:3]}')

    return complete.reset_index(drop=True)

In [ ]:
# ── 2-2. SVI → 도로 Map Matching ─────────────────────────────────────────────
def assign_svi_to_roads(svi_df, buffered_roads):
    """
    실제 SVI 촬영 좌표 기준 Map Matching.
    생성 point 좌표가 아닌 실제 SVI 좌표 사용 —
    촬영 차량이 실제로 위치한 도로에 이미지를 귀속.
    """
    svi_matched = spatial_join_points_to_roads(
        svi_df, buffered_roads,
        lon_col='svi_lon', lat_col='svi_lat', time_col=None
    )
    keep_cols = [c for c in ['location_id', 'date', 'LINK_ID',
                              'back', 'front', 'left', 'right',
                              'distance_to_line'] if c in svi_matched.columns]
    return svi_matched[keep_cols].reset_index(drop=True)

In [ ]:
# ── 2-3. Gemma 4 캡션 생성 ────────────────────────────────────────────────────
GEMMA_SYSTEM_PROMPT = """You are an urban transportation expert analyzing Korean \
street-level imagery to support illegal parking prediction research.
Your task is to describe physical road environments that are NOT captured by administrative \
databases such as official zoning maps, road width records, or POI counts.
Write only about what is clearly and directly observable iㅊn the images. \
Simply omit anything that is unclear or not identifiable.
Write in fluent, descriptive prose sentences."""

def make_gemma_prompt(loc_id=None, date=None):
    return """The following 4 images show the same road location captured from 4 directions:
Image 1 (Back) · Image 2 (Front) · Image 3 (Left) · Image 4 (Right)

Synthesize all 4 views and write TWO short paragraphs (3–5 sentences each).

[Paragraph 1 — Road function]
Describe the overall character of this road environment: the types of buildings,
businesses, or land uses present (e.g., dense commercial strip, residential neighborhood,
school zone, office district, mixed-use, green space). Focus on the actual street-level
atmosphere, not what official zoning might say. Note whether this feels like a major
arterial road or a narrow side street.

[Paragraph 2 — Illegal parking environment]
Describe the clearly visible features relevant to illegal parking. Cover whether yellow
no-parking lines (solid line = no stopping, dashed line = no parking) or no-parking signs
are present and legible — including any Korean-language road signs or painted markings.
Describe how the sidewalk is separated from the road, if at all (e.g., curb, bollards,
guardrail, planter, or no separation). Note whether there are designated on-street parking
spaces marked on the road surface, and whether any vehicles are currently parked or stopped
along the roadside. If a public parking lot entrance, bus stop, bus lane markings, or
subway station entrance is visible, include that as well. Finally, estimate the road width
based on the number of lanes visible."""


def generate_captions_gemma4(svi_matched, output_path=CAPTION_OUTPUT,
                              api_key=GOOGLE_API_KEY, model_id=GEMMA_MODEL,
                              rpm_limit=GEMMA_RPM_LIMIT):
    """
    Gemma 4 API: 4방향 이미지 동시 입력 → 지점당 캡션 1개 생성.
    체크포인트 저장으로 중단 후 재시작 가능.
    """
    if not api_key:
        raise ValueError('GOOGLE_API_KEY가 설정되지 않았습니다.')

    client   = genai.Client(api_key=api_key)
    interval = 60.0 / rpm_limit

    if Path(output_path).exists():
        with open(output_path, 'r', encoding='utf-8') as f:
            results = json.load(f)
        done_ids = {r['location_id'] for r in results}
        print(f'📂 체크포인트 로드: {len(done_ids)}개 기처리')
    else:
        results, done_ids = [], set()

    remaining = svi_matched[~svi_matched['location_id'].isin(done_ids)]
    print(f'🔄 처리 예정: {len(remaining):,}개 / 전체 {len(svi_matched):,}개')

    for i, row in enumerate(remaining.itertuples(index=False)):
        loc_id  = row.location_id
        date    = row.date
        link_id = row.LINK_ID

        contents = []
        for direction in ['back', 'front', 'left', 'right']:
            img_path = getattr(row, direction, None)
            if img_path and Path(img_path).exists():
                with open(img_path, 'rb') as f:
                    img_bytes = f.read()
                contents.append(
                    types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'))
        contents.append(make_gemma_prompt(loc_id, date))

        caption = None
        for attempt in range(3):
            try:
                resp = client.models.generate_content(
                    model=model_id, contents=contents,
                    config=types.GenerateContentConfig(
                        system_instruction=GEMMA_SYSTEM_PROMPT,
                        temperature=0.2, max_output_tokens=1024))
                caption = resp.text
                usage   = resp.usage_metadata
                results.append({
                    'location_id'  : loc_id, 'date': date,
                    'LINK_ID'      : int(link_id), 'caption': caption,
                    'input_tokens' : usage.prompt_token_count if usage else 0,
                    'output_tokens': usage.candidates_token_count if usage else 0})
                break
            except Exception as e:
                print(f'  ⚠️ {loc_id} | 시도 {attempt+1}/3 | {e}')
                if attempt < 2: time.sleep(2 ** attempt)

        if caption is None:
            print(f'  ❌ {loc_id} 최종 실패')

        if (i + 1) % 50 == 0:
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f'  💾 {i+1}/{len(remaining)} 저장')

        time.sleep(interval)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'\n✅ 캡션 생성 완료: {sum(1 for r in results if r.get("caption")):,}개')
    return results

In [ ]:
# ── 2-4. SBERT 임베딩 ────────────────────────────────────────────────────────
def compute_sbert_embeddings(captions, model_name=SBERT_MODEL,
                              output_path=EMBEDDING_OUTPUT):
    """캡션 텍스트 → all-mpnet-base-v2 임베딩 (768차원, 코사인 유사도용 정규화)."""
    print(f'📥 SBERT 모델 로드: {model_name}')
    sbert = SentenceTransformer(model_name)

    valid   = [r for r in captions if r.get('caption')]
    texts   = [r['caption'] for r in valid]
    meta_df = pd.DataFrame(valid)[['location_id', 'LINK_ID', 'date']]

    print(f'🔢 임베딩 생성 중: {len(texts):,}개 → 768차원')
    embeddings = sbert.encode(texts, batch_size=32,
                               show_progress_bar=True,
                               normalize_embeddings=True)  # 코사인 = 내적으로 계산 가능
    np.save(output_path, embeddings)
    print(f'✅ 임베딩 완료: {embeddings.shape} | 저장: {output_path}')
    return embeddings, meta_df

In [ ]:
# ── 2-5. Point-level Similarity 계산 ─────────────────────────────────────────
def compute_point_level_similarity(embeddings, meta_df, all_link_ids,
                                    output_path=SIM_MATRIX_OUTPUT):
    """
    Symmetric Mean of Row-max:
        sim(A, B) = ( mean_i[max_j S_ij] + mean_j[max_i S_ij] ) / 2
    → 도로 길이(지점 수) 차이에 강건, 정보 손실 최소화.
    SVI 없는 도로는 유사도 0 (adj_visual 연결 없음).
    """
    meta_df = meta_df.reset_index(drop=True)
    link_to_indices = defaultdict(list)
    for idx, row in meta_df.iterrows():
        link_to_indices[int(row['LINK_ID'])].append(idx)

    N_total = len(all_link_ids)
    roads_with_svi = [lid for lid in all_link_ids if lid in link_to_indices]
    print(f'ℹ️  SVI 보유: {len(roads_with_svi):,} / 전체: {N_total:,}개')

    # 전체 지점 쌍 유사도 1회 계산 (정규화 임베딩이므로 내적 = 코사인)
    print('🔢 지점 간 유사도 행렬 계산 중...')
    S_all = embeddings @ embeddings.T
    np.fill_diagonal(S_all, 0)

    lid_to_pos = {lid: pos for pos, lid in enumerate(all_link_ids)}
    sim_matrix = np.zeros((N_total, N_total), dtype=np.float32)

    for i, lid_i in enumerate(roads_with_svi):
        idx_i = link_to_indices[lid_i]
        pos_i = lid_to_pos[lid_i]
        for j, lid_j in enumerate(roads_with_svi):
            if j <= i: continue
            idx_j  = link_to_indices[lid_j]
            pos_j  = lid_to_pos[lid_j]
            S_block = S_all[np.ix_(idx_i, idx_j)]
            sim_ij  = (S_block.max(axis=1).mean() + S_block.max(axis=0).mean()) / 2
            sim_matrix[pos_i, pos_j] = sim_ij
            sim_matrix[pos_j, pos_i] = sim_ij
        if (i + 1) % 200 == 0:
            print(f'  진행: {i+1}/{len(roads_with_svi)}')

    np.save(output_path, sim_matrix)
    print(f'✅ 유사도 행렬: {sim_matrix.shape} | 저장: {output_path}')
    return sim_matrix

---
## SECTION 3 : 그래프 구성
`adj_phys` (위상 인접), `adj_adapt` (컨텍스트 유사도), `adj_visual` (SVI 시각 유사도).

adj_adapt와 adj_visual 모두 **임계치 기반** 희소화 적용 (일관된 구성 원칙).

In [ ]:
# ── 3-0. top-k 기반 인접 행렬 공통 생성 함수 ─────────────────────────────────────────
def build_adj_threshold_topk(sim_matrix, threshold, k, label='', device='cuda'):
    """
    Two-Stage Filtering: Threshold → Top-K → Softmax 정규화

    [수정] threshold 미만 값을 0으로 바꾸고 softmax하면
           e^0 = 1이 되어 threshold 미만 연결에도 가중치가 부여됨.
           softmax 전까지 -inf를 유지하고, 이후에만 0으로 처리.
    """
    sim = (torch.FloatTensor(sim_matrix).to(device)
           if not isinstance(sim_matrix, torch.Tensor) else sim_matrix.to(device))
    N = sim.size(0)

    # Step 1: Threshold 마스킹 (-inf 유지)
    sim_masked = sim.clone()
    sim_masked[sim < threshold] = float('-inf')
    sim_masked.fill_diagonal_(float('-inf'))

    # Step 2: Top-K 선택 (k가 N보다 클 경우 clamp)
    k_actual = min(k, N - 1)
    top_k_val, top_k_idx = torch.topk(sim_masked, k_actual, dim=-1)  # (N, k)
    valid_mask = ~torch.isinf(top_k_val)  # threshold 통과한 연결만 True

    # Step 3: Softmax — -inf는 그대로 넘겨서 자동으로 0이 됨
    # nan_to_num으로 -inf→0 안전 처리 (all-inf 행 = 연결 없는 노드)
    weights = F.softmax(top_k_val, dim=-1)          # -inf → 0, 나머지 정규화
    weights = torch.nan_to_num(weights, nan=0.0)    # all-inf 행: nan → 0
    weights[~valid_mask] = 0.0                       # 명시적 0 보장

    # Step 4: Sparse 텐서 구성
    row_idx = torch.arange(N, device=device).unsqueeze(1).expand_as(top_k_idx).reshape(-1)
    col_idx = top_k_idx.reshape(-1)
    values  = weights.reshape(-1)

    nonzero_mask = values > 0
    row_idx = row_idx[nonzero_mask]
    col_idx = col_idx[nonzero_mask]
    values  = values[nonzero_mask]

    adj_sparse = torch.sparse_coo_tensor(
        indices=torch.stack([row_idx, col_idx]),
        values=values,
        size=(N, N),
        device=device
    ).to_sparse_csr()

    avg_deg          = values.size(0) / N
    actual_k         = valid_mask.sum(dim=1)
    isolated_nodes   = (actual_k == 0).sum().item()

    print(f'\n✅ adj_{label} (threshold={threshold:.2f}, top-k={k_actual}, device={device}):')
    print(f'   평균 연결 수: {avg_deg:.2f}개')
    print(f'   (min={actual_k.min()}, max={actual_k.max()}, '
          f'mean={actual_k.float().mean():.2f})')
    print(f'   연결 없는 노드(SVI 없음 등): {isolated_nodes}개')
    print(f'   Sparse shape: {adj_sparse.shape} | 비영 원소: {values.size(0):,}개')

    return adj_sparse

In [ ]:
# ── 3-1. adj_phys (위상 인접) ─────────────────────────────────────────────────
def build_adj_phys(buffered_roads, pivot_df, r=1.0):
    """도로 시작점/끝점 근접성(r미터)으로 위상 인접 그래프 구성."""
    road_gdf_clean = (buffered_roads[['LINK_ID', 'geometry_line']]
                      .copy().set_geometry('geometry_line').reset_index(drop=True))
    link_ids_list  = road_gdf_clean['LINK_ID'].tolist()

    endpoint_records = []
    for i, row in road_gdf_clean.iterrows():
        coords = list(row['geometry_line'].coords)
        endpoint_records += [
            {'road_idx': i, 'LINK_ID': row['LINK_ID'], 'geometry': Point(coords[0][:2])},
            {'road_idx': i, 'LINK_ID': row['LINK_ID'], 'geometry': Point(coords[-1][:2])},
        ]

    endpoint_gdf = gpd.GeoDataFrame(endpoint_records, geometry='geometry',
                                     crs=road_gdf_clean.crs)
    endpoint_gdf['buffer'] = endpoint_gdf.geometry.buffer(r)

    joined = gpd.sjoin(
        endpoint_gdf.set_geometry('buffer')[['road_idx', 'LINK_ID', 'buffer']],
        road_gdf_clean[['LINK_ID', 'geometry_line']].rename(
            columns={'LINK_ID': 'LINK_ID_other'}),
        how='inner', predicate='intersects')

    edges = [(row['LINK_ID'], row['LINK_ID_other'])
             for _, row in joined.iterrows()
             if row['LINK_ID'] != row['LINK_ID_other']]

    G = nx.Graph()
    G.add_nodes_from(link_ids_list); G.add_edges_from(edges)
    G.remove_edges_from(nx.selfloop_edges(G))
    components = list(nx.connected_components(G)) # Define components here
    print(f'  그래프: 노드 {G.number_of_nodes():,} | 엣지 {G.number_of_edges():,} | '
          f'평균 차수 {sum(dict(G.degree()).values())/G.number_of_nodes():,.2f} | '
          f'연결 그래프? {nx.is_connected(G)} | 컴포넌트 수: {len(components)}') # Corrected format specifier and removed extra comma

    link_to_idx = {link: idx for idx, link in enumerate(pivot_df.index.tolist())}
    edge_list   = []
    for u, v in G.edges():
        iu, iv = link_to_idx.get(u), link_to_idx.get(v)
        if iu is not None and iv is not None:
            edge_list += [[iu, iv], [iv, iu]]

    N_nodes    = len(pivot_df.index)
    edge_index = torch.tensor(edge_list, dtype=torch.long).T
    adj_phys   = torch.zeros(N_nodes, N_nodes)
    adj_phys[edge_index[0], edge_index[1]] = 1.0
    adj_phys   = adj_phys / adj_phys.sum(dim=-1, keepdim=True).clamp(min=1)

    print(f'✅ adj_phys: {tuple(adj_phys.shape)} | 비영 원소 {int((adj_phys > 0).sum()):,}')
    return adj_phys, G

---
## SECTION 4 : Dataset
기존 코드와 동일.

In [ ]:
# ── 4-1. CSTDataset ───────────────────────────────────────────────────────────
class CSTDataset(Dataset):
    """
    한 샘플 = 예측 시작 시점 t의 (x_recent, x_daily, x_weekly, temporal, target)
    target: (N, H_HORIZON)  ← 시점 t부터 t+H-1까지 '미래 H시간'
    """
    def __init__(self, pivot_df, temporal_context, start_date, end_date):
        all_times  = pivot_df.columns
        n_times    = len(all_times)                       # [신규] 전체 시점 개수
        start, end = pd.to_datetime(start_date), pd.to_datetime(end_date)
        min_offset = 168 + H_WEEKLY

        self.valid_indices = [
            i for i, t in enumerate(all_times)
            if start <= t <= end                          # 기준 시점이 구간 안
            and i >= min_offset                           # 과거 윈도우 확보 (기존)
            and i + H_HORIZON <= n_times                  # [신규] 미래 H칸이 배열 안 (에러 방지)
            and all_times[i + H_HORIZON - 1] <= end       # [신규] 미래 H칸이 구간 안 (누수 방지)
        ]
        assert len(self.valid_indices) > 0, f'유효 샘플 없음 ({start_date} ~ {end_date})'
        self.data     = pivot_df.values.astype(np.float32)
        self.temporal = temporal_context.astype(np.float32)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        t        = self.valid_indices[idx]
        x_recent = np.log1p(self.data[:, t - H_RECENT : t])           # (기존 그대로)
        x_daily  = np.log1p(self.data[:, t - 24  - H_DAILY  : t - 24])
        x_weekly = np.log1p(self.data[:, t - 168 - H_WEEKLY : t - 168])
        target   = self.data[:, t : t + H_HORIZON]                    # [수정] (N,) → (N, H)
        return (torch.FloatTensor(x_recent).unsqueeze(0),
                torch.FloatTensor(x_daily).unsqueeze(0),
                torch.FloatTensor(x_weekly).unsqueeze(0),
                torch.FloatTensor(self.temporal[t]),
                torch.FloatTensor(target))                            # [수정] 이제 (N, H)

# ── 4-2. DataLoader 빌더 ─────────────────────────────────────────────────────
def build_dataloaders(pivot_df, temporal_context,
                      train_start, train_end, val_start, val_end,
                      test_start, test_end, batch_size=BATCH_SIZE, num_workers=0):
    def make(s, e):
        ds = CSTDataset(pivot_df, temporal_context, s, e)
        return DataLoader(ds, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)

    loaders = [make(train_start, train_end),
               make(val_start, val_end),
               make(test_start, test_end)]
    for name, dl in zip(['Train', 'Val', 'Test'], loaders):
        print(f'  {name}: {len(dl.dataset):,}샘플')
    return loaders

---
## SECTION 5 : 모델
**변경사항:**
- `STBlock`: `gcn_visual` 추가, GCN 출력 단순합 → 학습 가능한 가중합 (α·phys + β·adapt + γ·visual)
- `MCAST_ZINB.forward`: `adj_visual` 파라미터 추가

In [ ]:
# ── 5-1. TCN 입력 준비 ────────────────────────────────────────────────────────
def prepare_tcn_input(x, temporal, static_context, keyword_context):   # [keyword_context 추가]
    """traffic + temporal + static_context + keyword_context → (B, C, N, W)"""
    B, _, N, W = x.shape
    D_s = static_context.size(-1)
    D_k = keyword_context.size(-1)                                      # [신규]
    t_  = temporal.view(B, 4, 1, 1).expand(B, 4, N, W)
    s_  = static_context.T.unsqueeze(0).unsqueeze(-1).expand(B, D_s, N, W)
    k_  = keyword_context.T.unsqueeze(0).unsqueeze(-1).expand(B, D_k, N, W)  # [신규]
    return torch.cat([x, t_, s_, k_], dim=1)                           # [k_ 추가]

# ── 5-2. Temporal Gated Fusion (기존 유지) ────────────────────────────────────
class TemporalGatedFusion(nn.Module):
    """
    Recent/Daily/Weekly 스트림을 노드별 gate로 동적 융합.
    시점에 따라 어느 주기가 중요한지 데이터가 결정 → 정적 가중합과 구별.
    """
    def __init__(self, dim):
        super().__init__()
        self.gate_layer = nn.Sequential(nn.Linear(dim * 3, 3), nn.Softmax(dim=-1))

    def forward(self, h_r, h_d, h_w):
        gates         = self.gate_layer(torch.cat([h_r, h_d, h_w], dim=-1))
        g_r, g_d, g_w = gates.unbind(dim=-1)
        return (g_r.unsqueeze(-1) * h_r +
                g_d.unsqueeze(-1) * h_d +
                g_w.unsqueeze(-1) * h_w)

In [ ]:
# ── 5-3. STBlock (gcn_visual + 학습 가능 가중합 추가) ─────────────────────────
class STBlock(nn.Module):
    """
    Spatial-Temporal Block (B안 적용)

    [B안 핵심]
    - 각 블록이 (h_r_out, h_d_out, h_w_out) 을 반환
    - GCN 공간 정보를 세 스트림 각각에 residual로 주입
    - 다음 블록의 tcn_r/tcn_d/tcn_w 가 서로 다른 스트림을 계속 처리
    → 블록 2~4에서도 recent/daily/weekly 패턴이 독립적으로 유지됨

    [기존 문제]
    h_r = h_d = h_w = h 로 동일하게 설정해버려서
    2번째 블록부터 3개 TCN이 완전히 동일한 입력을 받음
    → Temporal Gated Fusion 의미 소실
    """
    def __init__(self, in_dim, out_dim, dilation):
        super().__init__()
        _conv = lambda: nn.Conv2d(in_dim, out_dim,
                                   kernel_size=(1, 3), dilation=(1, dilation),
                                   padding=(0, dilation))
        self.tcn_r = _conv(); self.tcn_d = _conv(); self.tcn_w = _conv()
        self.fusion      = TemporalGatedFusion(out_dim)
        self.gcn_phys    = nn.Linear(out_dim, out_dim)
        self.gcn_adapt   = nn.Linear(out_dim, out_dim)
        self.gcn_visual  = nn.Linear(out_dim, out_dim)
        self.gcn_weights = nn.Parameter(torch.ones(3) / 3)  # α, β, γ
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x_r, x_d, x_w, adj_phys, adj_adapt, adj_visual):
        # 1) 스트림별 TCN — 시퀀스 전체 보존 (B, D, N, T)
        h_r_seq = F.relu(self.tcn_r(x_r))
        h_d_seq = F.relu(self.tcn_d(x_d))
        h_w_seq = F.relu(self.tcn_w(x_w))

        # 2) 마지막 시점 → Temporal Gated Fusion
        h_r = h_r_seq[..., -1].permute(0, 2, 1)   # (B, N, D)
        h_d = h_d_seq[..., -1].permute(0, 2, 1)
        h_w = h_w_seq[..., -1].permute(0, 2, 1)
        h_fused = self.fusion(h_r, h_d, h_w)       # (B, N, D)
        B = h_fused.size(0)

        # 3) Triple-path GCN
        out_phys = torch.matmul(adj_phys, self.gcn_phys(h_fused))

        gcn_in_adapt = self.gcn_adapt(h_fused)
        out_adapt = torch.stack(
            [torch.sparse.mm(adj_adapt, gcn_in_adapt[b]) for b in range(B)])

        gcn_in_visual = self.gcn_visual(h_fused)
        out_visual = torch.stack(
            [torch.sparse.mm(adj_visual, gcn_in_visual[b]) for b in range(B)])

        # 4) 학습 가능한 가중합
        w = F.softmax(self.gcn_weights, dim=0)
        h_gcn = F.relu(self.norm(
            w[0]*out_phys + w[1]*out_adapt + w[2]*out_visual))  # (B, N, D)

        # 5) [B안 핵심] GCN 정보를 세 스트림 각각에 residual 주입
        #    → 다음 블록의 tcn_r/d/w가 서로 다른 시퀀스를 받게 됨
        h_gcn_t  = h_gcn.permute(0, 2, 1).unsqueeze(-1)  # (B, D, N, 1)
        h_r_out  = h_r_seq + h_gcn_t                     # (B, D, N, T)
        h_d_out  = h_d_seq + h_gcn_t
        h_w_out  = h_w_seq + h_gcn_t

        return h_r_out, h_d_out, h_w_out

In [ ]:
# ── 5-4. MCAST-ZINB 전체 모델 ──────────────────────────────────────────────────
class MCAST_ZINB(nn.Module):
    def __init__(self, static_dim, keyword_dim, n_layers=N_LAYERS, hidden_dim=HIDDEN_DIM):  # [keyword_dim 추가]
        super().__init__()
        first_in_dim = 1 + 4 + static_dim + keyword_dim               # [keyword_dim 추가]
        self.blocks = nn.ModuleList([
            STBlock(first_in_dim if i == 0 else hidden_dim, hidden_dim, dilation=2**i)
            for i in range(n_layers)
        ])
        self.pi_head    = nn.Linear(hidden_dim, H_HORIZON)   # [수정] 1 → H_HORIZON
        self.mu_head    = nn.Linear(hidden_dim, H_HORIZON)   # [수정]
        self.theta_head = nn.Linear(hidden_dim, H_HORIZON)   # [수정]

    def forward(self, x_r, x_d, x_w, temporal, static_context, keyword_context,
            adj_phys, adj_adapt, adj_visual):
        x_r_in = prepare_tcn_input(x_r, temporal, static_context, keyword_context)
        x_d_in = prepare_tcn_input(x_d, temporal, static_context, keyword_context)
        x_w_in = prepare_tcn_input(x_w, temporal, static_context, keyword_context)
        h_r, h_d, h_w = x_r_in, x_d_in, x_w_in

        for block in self.blocks:
            h_r, h_d, h_w = block(h_r, h_d, h_w, adj_phys, adj_adapt, adj_visual)
            #  ↑ 튜플 언패킹으로 수정 (기존: h = block(...); h_r = h_d = h_w = h)

        # 세 스트림 평균 후 마지막 시점 사용
        h_final = ((h_r + h_d + h_w) / 3)[..., -1].permute(0, 2, 1)  # (B, N, D)
        #          ↑ 기존: h[..., -1] — h가 튜플이라 에러

        pi    = torch.sigmoid(self.pi_head(h_final))      # [수정] squeeze 제거 → (B, N, H)
        mu    = F.softplus(self.mu_head(h_final))         # [수정]
        theta = torch.exp(self.theta_head(h_final))       # [수정]
        return pi, mu, theta


# ── 5-5. ZINB Loss (기존 유지) ────────────────────────────────────────────────
def zinb_loss(y, pi, mu, theta, weight=None, eps=1e-8):
    nb_log_prob = (
        torch.lgamma(y+theta) - torch.lgamma(theta) - torch.lgamma(y+1)
        + theta * (torch.log(theta+eps) - torch.log(mu+theta+eps))
        + y     * (torch.log(mu+eps)    - torch.log(mu+theta+eps))
    )
    log_prob = torch.where(
        y < 1e-8,
        torch.log(pi + (1-pi)*torch.exp(nb_log_prob) + eps),
        torch.log(1-pi+eps) + nb_log_prob
    )
    if weight is not None:
        return -(log_prob * weight).sum() / (weight.sum() + eps)
    return -log_prob.mean()

---
## SECTION 6 : 학습 및 평가
모든 함수에 `adj_visual` 파라미터 추가. 나머지 로직은 기존과 동일.

In [ ]:
# ── 6-1. 노드별 극단값 threshold 계산 (기존 유지) ────────────────────────────
def compute_extreme_threshold_per_node(train_loader, percentile=0.9, device=device):
    """학습 세트 기준 노드별 극단값 threshold (학습 전 1회 실행)."""
    _, _, _, _, target = next(iter(train_loader))
    N, node_values = target.shape[1], [[] for _ in range(target.shape[1])]

    for _, _, _, _, target in train_loader:
        y = target.to(device)
        for n in range(N):
            yn = y[:, n, :]                    # [수정] (B, H) — 이 노드의 모든 배치·모든 horizon
            nz = yn[yn > 1e-8]                 # 0이 아닌 값만 (1차원으로 평탄화됨)
            if len(nz) > 0: node_values[n].append(nz.cpu())

    threshold = torch.zeros(N, device=device)
    for n in range(N):
        if not node_values[n]:
            threshold[n] = 1.0
        else:
            vals = torch.cat(node_values[n])
            threshold[n] = (torch.quantile(vals, percentile).to(device)
                            if len(vals) >= 10 else vals.max().to(device) * 0.5)
    print(f'✅ threshold: mean={threshold.mean():.2f}, max={threshold.max():.2f}')
    return threshold

In [ ]:
# ── 6-2. 학습 (adj_visual 추가) ──────────────────────────────────────────────
def train_one_epoch(model, train_loader, optimizer,
                    static_context, keyword_context,                    # [keyword_context 추가]
                    adj_phys, adj_adapt, adj_visual,
                    threshold_per_node=None, warmup=True,
                    alpha=5.0, beta=5.0, entropy_reg=0.0001,
                    accumulation_steps=2, normalize_weight=True, device=device):
    model.train()
    total_loss = 0.0; optimizer.zero_grad()

    for step, (x_r, x_d, x_w, temporal, target) in enumerate(train_loader):
        x_r, x_d, x_w = x_r.to(device), x_d.to(device), x_w.to(device)
        temporal, y    = temporal.to(device), target.to(device)

        pi, mu, theta = model(x_r, x_d, x_w, temporal,
                               static_context, keyword_context,         # [수정]
                               adj_phys, adj_adapt, adj_visual)

        # 이하 동일 (loss 계산 등 변경 없음)
        if warmup:
            weight = torch.where(y > 1e-8, torch.ones_like(y), torch.full_like(y, 0.05))
        else:
            normalized     = torch.clamp(
                y / (threshold_per_node.view(1, -1, 1) + 1e-8), 0, 2)   # [수정] unsqueeze(0) → view(1,-1,1)
            extreme_weight = 1.0 + beta * torch.relu(normalized - 1.0)
            if normalize_weight:
                extreme_weight /= (
                    extreme_weight.mean(dim=(0, 2), keepdim=True).detach() + 1e-8)  # [수정] mean(0) → mean((0,2))
            weight = torch.where(y < 1e-8, torch.ones_like(y), alpha * extreme_weight)

        eps        = 1e-8
        loss       = zinb_loss(y, pi, mu, theta, weight=weight)
        pi_entropy = -(pi*torch.log(pi+eps) + (1-pi)*torch.log(1-pi+eps)).mean()
        loss = loss + entropy_reg * pi_entropy

        total_loss += loss.item()
        loss = loss / accumulation_steps
        loss.backward()

        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step(); optimizer.zero_grad()

    if len(train_loader) % accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step(); optimizer.zero_grad()

    return total_loss / len(train_loader)

In [ ]:
@torch.no_grad()
def evaluate(model, loader, static_context, keyword_context,
             adj_phys, adj_adapt, adj_visual, device=device):
    """전체 평균 지표 (val/조기종료용). (B,N,H)에서 동작, 원래 지표 모두 유지."""
    model.eval()
    all_pi, all_mu, all_y = [], [], []
    total_loss = 0.0

    for x_r, x_d, x_w, temporal, target in loader:
        x_r, x_d, x_w = x_r.to(device), x_d.to(device), x_w.to(device)
        temporal, y = temporal.to(device), target.to(device)
        pi, mu, theta = model(x_r, x_d, x_w, temporal,
                              static_context, keyword_context,
                              adj_phys, adj_adapt, adj_visual)
        total_loss += zinb_loss(y, pi, mu, theta).item()
        all_pi.append(pi.cpu()); all_mu.append(mu.cpu()); all_y.append(y.cpu())

    # (T_eval, N, H) 로 쌓기
    all_pi = torch.cat(all_pi, dim=0).numpy()
    all_mu = torch.cat(all_mu, dim=0).numpy()
    all_y  = torch.cat(all_y,  dim=0).numpy()

    return _compute_all_metrics(all_pi, all_mu, all_y, avg_loss=total_loss / len(loader))

def _compute_all_metrics(all_pi, all_mu, all_y, avg_loss=None):
    """
    입력이 (T,N) 이든 (T,N,H) 이든 동작.
    원래 evaluate가 계산하던 지표를 모두 계산:
    zero_rate(true/pred), kl_div, mae, rmse, mae_pos, rmse_pos,
    recall_pos, precision_pos, f1_pos, hr_at_20
    HR@20은 '시점별 노드 랭킹'이라 (시점, 노드) 축이 필요 →
    (T,N,H)면 H를 시점 축으로 펼쳐서 (T*H, N)으로 만들어 계산.
    """
    # ── HR@20 계산용: (관측단위, N) 형태로 정리 ──
    if all_y.ndim == 3:                       # (T, N, H)
        T, N, H = all_y.shape
        expected_3d = (1 - all_pi) * all_mu   # (T,N,H)
        # 시점축으로 H를 펼침: 각 (t,h)가 하나의 "예측 시점 슬라이스"
        y_2d  = np.transpose(all_y,       (0, 2, 1)).reshape(T * H, N)   # (T*H, N)
        e_2d  = np.transpose(expected_3d, (0, 2, 1)).reshape(T * H, N)
    else:                                     # (T, N)  — 단일스텝 호환
        T, N = all_y.shape
        expected_3d = (1 - all_pi) * all_mu
        y_2d = all_y
        e_2d = expected_3d

    y_flat        = y_2d.flatten()
    expected_flat = e_2d.flatten()
    pred_flat     = (expected_flat > 0.5).astype(int)

    # ── 학습 진단 ──
    zero_rate_true = np.mean(y_flat == 0)
    zero_rate_pred = np.mean(expected_flat < 0.5)

    y_sum, exp_sum = y_flat.sum(), expected_flat.sum()
    if y_sum > 0 and exp_sum > 0:
        p = y_flat / y_sum + 1e-10
        q = expected_flat / exp_sum + 1e-10
        kl_div = float(np.sum(p * np.log(p / q)))
    else:
        kl_div = float('nan')

    # ── 회귀 지표 ──
    mae  = mean_absolute_error(y_flat, expected_flat)
    rmse = np.sqrt(mean_squared_error(y_flat, expected_flat))
    pos_mask = y_flat > 0
    if pos_mask.sum() > 0:
        mae_pos  = mean_absolute_error(y_flat[pos_mask], expected_flat[pos_mask])
        rmse_pos = np.sqrt(mean_squared_error(y_flat[pos_mask], expected_flat[pos_mask]))
    else:
        mae_pos = rmse_pos = float('nan')

    # ── 분류 지표 (발생 여부) ──
    true_bin = (y_flat > 0).astype(int)
    pred_bin = pred_flat
    tp = ((pred_bin == 1) & (true_bin == 1)).sum()
    fp = ((pred_bin == 1) & (true_bin == 0)).sum()
    fn = ((pred_bin == 0) & (true_bin == 1)).sum()
    recall    = tp / (tp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    # ── 랭킹 지표: HR@20 (관측단위별 상위 20%) ──
    n_slice, N = y_2d.shape
    k = max(1, int(np.floor(0.2 * N)))
    hits = events = 0
    for t in range(n_slice):
        true_nodes = np.where(y_2d[t] > 0)[0]
        if true_nodes.size == 0:
            continue
        events += len(true_nodes)
        top_k = np.argsort(e_2d[t])[::-1][:k]
        hits += np.intersect1d(top_k, true_nodes).size
    hr_at_20 = hits / events if events > 0 else float('nan')

    out = {
        'zero_rate_true': zero_rate_true, 'zero_rate_pred': zero_rate_pred,
        'kl_div': kl_div,
        'mae': mae, 'rmse': rmse, 'mae_pos': mae_pos, 'rmse_pos': rmse_pos,
        'recall_pos': recall, 'precision_pos': precision, 'f1_pos': f1,
        'hr_at_20': hr_at_20,
    }
    if avg_loss is not None:
        out['loss'] = avg_loss
    return out

In [ ]:
@torch.no_grad()
def evaluate_multistep(model, loader, static_context, keyword_context,
                       adj_phys, adj_adapt, adj_visual, device=device):
    """horizon(1..H)별로 '모든 지표'를 분해 + 전체 평균도 함께 반환."""
    model.eval()
    all_pi, all_mu, all_y = [], [], []
    total_loss = 0.0

    for x_r, x_d, x_w, temporal, target in loader:
        x_r, x_d, x_w = x_r.to(device), x_d.to(device), x_w.to(device)
        temporal, y = temporal.to(device), target.to(device)
        pi, mu, theta = model(x_r, x_d, x_w, temporal,
                              static_context, keyword_context,
                              adj_phys, adj_adapt, adj_visual)
        total_loss += zinb_loss(y, pi, mu, theta).item()
        all_pi.append(pi.cpu()); all_mu.append(mu.cpu()); all_y.append(y.cpu())

    all_pi = torch.cat(all_pi, dim=0).numpy()   # (T, N, H)
    all_mu = torch.cat(all_mu, dim=0).numpy()
    all_y  = torch.cat(all_y,  dim=0).numpy()
    H = all_y.shape[-1]

    # horizon별: h를 하나 고정 → (T, N) 슬라이스로 만들어 동일한 지표 계산
    per_horizon = []
    for h in range(H):
        m = _compute_all_metrics(all_pi[:, :, h], all_mu[:, :, h], all_y[:, :, h])
        m['h'] = h + 1
        per_horizon.append(m)

    # 전체 평균 (모든 horizon 합산)
    overall = _compute_all_metrics(all_pi, all_mu, all_y, avg_loss=total_loss / len(loader))
    return per_horizon, overall

In [ ]:
# ── 6-3b. 메트릭 출력 함수 ────────────────────────────────────────────────────
def print_metrics(metrics, split='Val'):
    """평가 메트릭을 카테고리별로 출력."""
    print(f"\n── {split} 평가 결과 ─────────────────────────────────────")
    print(f"  [학습 진단]")
    print(f"  Loss             : {metrics['loss']:.4f}")
    print(f"  Zero rate (실제) : {metrics['zero_rate_true']:.4f}")
    print(f"  Zero rate (예측) : {metrics['zero_rate_pred']:.4f}")
    print(f"  KL Divergence    : {metrics['kl_div']:.4f}")
    print(f"  [회귀 지표]")
    print(f"  MAE              : {metrics['mae']:.4f}")
    print(f"  RMSE             : {metrics['rmse']:.4f}")
    print(f"  MAE@positive     : {metrics['mae_pos']:.4f}")
    print(f"  RMSE@positive    : {metrics['rmse_pos']:.4f}")
    print(f"  [분류 지표]")
    print(f"  Recall@positive  : {metrics['recall_pos']:.4f}")
    print(f"  Precision@pos    : {metrics['precision_pos']:.4f}")
    print(f"  F1@positive      : {metrics['f1_pos']:.4f}")
    print(f"  [랭킹 지표]")
    print(f"  HR@20            : {metrics['hr_at_20']:.4f}")

In [ ]:
# ── 6-4. 전체 학습 루프 ──────────────────────────────────────────────────────
def run_training(model, train_loader, val_loader,
                 static_context, keyword_context,                       # [keyword_context 추가]
                 adj_phys, adj_adapt, adj_visual,
                 n_epochs=EPOCHS, lr=LR, warmup_epochs=5,
                 device=device, save_path='best_model.pt'):
    optimizer  = optim.Adam(model.parameters(), lr=lr)
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
    threshold  = compute_extreme_threshold_per_node(train_loader, device=device)
    best_loss  = float('inf'); history = []

    for epoch in range(1, n_epochs + 1):
        is_warmup  = (epoch <= warmup_epochs)
        train_loss = train_one_epoch(
            model, train_loader, optimizer,
            static_context, keyword_context,                            # [수정]
            adj_phys, adj_adapt, adj_visual,
            threshold_per_node=threshold, warmup=is_warmup, device=device)
        val_m = evaluate(model, val_loader,
                         static_context, keyword_context,               # [수정]
                         adj_phys, adj_adapt, adj_visual, device=device)
        scheduler.step(val_m['loss'])
        history.append({'epoch': epoch, 'train_loss': train_loss, **val_m})

        mode = 'warmup' if is_warmup else 'full '
        print(f'\n[Epoch {epoch}/{n_epochs} {mode}]')
        print(f'  Train Loss: {train_loss:.4f}')
        print_metrics(val_m, split='Val')

        if val_m['loss'] < best_loss:
            best_loss = val_m['loss']
            torch.save(model.state_dict(), save_path)
            print(f'  💾 Best model saved')

    return history

---
## SECTION 7-A : SVI 전처리 — 최초 1회만 실행

> **⚠️ 이 섹션은 최초 1회만 실행하면 됩니다.**  
> 결과 파일(`svi_captions.json`, `svi_embeddings.npy`)이 저장된 이후에는  
> SECTION 7-B부터 바로 실행하세요.

| 셀 | 작업 | 소요 시간 |
|---|---|---|
| Step A-1 | SVI 메타데이터 로드 + Map Matching | 수 분 |
| Step A-2 | Gemma 4 캡션 생성 | **수십 분~수 시간** (4,976개 지점) |
| Step A-3 | SBERT 임베딩 | 수 분 |

In [ ]:
import os
import shutil

# Python 변수로 경로 조합
src_json = f'{DRIVE_BASE}/거리영상_50m/json (1)'
src_img  = f'{DRIVE_BASE}/거리영상_50m/images (1)'

# shutil로 복사 (Python이 경로 변수를 직접 처리)
print('JSON 복사 중...')
shutil.copytree(src_json, '/content/json_local', dirs_exist_ok=True)
print('이미지 복사 중...')
shutil.copytree(src_img,  '/content/img_local',  dirs_exist_ok=True)

# 경로 업데이트
SVI_JSON_DIR = '/content/json_local'
SVI_IMG_DIR  = '/content/img_local'
print('✅ 복사 완료')

In [ ]:
# ── Step A-1. SVI 메타데이터 로드 + Map Matching ─────────────────────────────
# ※ 이 결과는 SECTION 7-B에서도 재사용되므로 한 번 실행 후 보존하세요.
svi_df      = load_svi_metadata(SVI_JSON_DIR, SVI_IMG_DIR)
svi_matched = assign_svi_to_roads(svi_df, buffered_roads)
print(f'SVI-도로 매핑 완료: {len(svi_matched):,}개 지점')

In [ ]:
import json

with open(CAPTION_OUTPUT, 'r', encoding='utf-8') as f:
    data = json.load(f)

clean = [r for r in data if r.get('caption')]  # null 제거
print(f'전체: {len(data)}개 → 정리 후: {len(clean)}개 ({len(data)-len(clean)}개 제거)')

with open(CAPTION_OUTPUT, 'w', encoding='utf-8') as f:
    json.dump(clean, f, ensure_ascii=False, indent=2)

print('✅ 저장 완료')

In [ ]:
# ── Step A-2. Gemma 4 캡션 생성 ─────────────────────────────────────────────
# ※ 최초 1회만 실행. 이후에는 svi_captions.json을 자동 로드합니다.
# ※ 중간에 중단되어도 체크포인트가 저장되어 재실행 시 이어서 진행됩니다.
# ※ API 키가 올바르게 설정되었는지 먼저 확인하세요 (SECTION 0-4).

captions = generate_captions_gemma4(svi_matched)
print(f'캡션 저장 완료: {CAPTION_OUTPUT}')

In [ ]:
# 실패한 첫 번째 케이스 하나만 직접 테스트
row = svi_matched.iloc[0]  # ← 이것만 변경
# 1. 이미지 파일 존재 여부 확인
print("=== 이미지 경로 확인 ===")
for direction in ['back', 'front', 'left', 'right']:
    img_path = getattr(row, direction, None)
    exists = Path(img_path).exists() if img_path else False
    print(f'{direction}: {img_path}')
    print(f'         존재={exists}')

# 2. API 직접 호출해서 응답 상세 확인
print("\n=== API 응답 확인 ===")
contents = []
for direction in ['back', 'front', 'left', 'right']:
    img_path = getattr(row, direction, None)
    if img_path and Path(img_path).exists():
        with open(img_path, 'rb') as f:
            img_bytes = f.read()
        contents.append(types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'))

print(f'이미지 {len(contents)}장 로드됨')

contents.append(make_gemma_prompt(row.location_id, row.date))
client = genai.Client(api_key=GOOGLE_API_KEY)

resp = client.models.generate_content(
    model=GEMMA_MODEL, contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=GEMMA_SYSTEM_PROMPT,
        temperature=0.2, max_output_tokens=1024))

print(f'resp.text     : {repr(resp.text)}')
print(f'finish_reason : {resp.candidates[0].finish_reason if resp.candidates else "없음"}')
print(f'safety_ratings: {resp.candidates[0].safety_ratings if resp.candidates else "없음"}')

In [ ]:
# ── Step A-3. SBERT 임베딩 ───────────────────────────────────────────────────
# ※ 최초 1회만 실행. 이후에는 svi_embeddings.npy를 자동 로드합니다.

embeddings, meta_df = compute_sbert_embeddings(captions)
print(f'임베딩 저장 완료: {EMBEDDING_OUTPUT}')
print(f'임베딩 shape: {embeddings.shape}')

---
## SECTION 7-B : 학습 파이프라인 — 매번 실행

> **✅ 이 섹션부터 실행하세요** (SVI 전처리가 완료된 이후).  
> 캡션/임베딩/그래프 파일이 존재하면 자동으로 로드합니다.

실행 순서: Step 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9

In [ ]:
# ── Step 1. 데이터 로드 ──────────────────────────────────────────────────────
road_gdf    = gpd.read_file(ROAD_SHP).to_crs(epsg=5179)
parking_df  = pd.read_csv(PARKING_CSV)
landuse_gdf = gpd.read_file(LANDUSE_SHP).to_crs(epsg=5179)
poi_counts  = pd.read_csv(POI_CSV)

road_gdf = road_gdf[road_gdf['Shape_Leng'] >= 1].reset_index(drop=True)
print(f'도로 구간: {len(road_gdf):,}개 | 불법주정차: {len(parking_df):,}건')

In [ ]:
# ── Step 2. 도로 전처리 ──────────────────────────────────────────────────────
buffered_roads = build_buffered_roads(road_gdf)
buffered_roads = join_landuse_by_largest_overlap(buffered_roads, landuse_gdf)
buffered_roads = merge_poi_counts(buffered_roads, poi_counts)
all_link_ids   = buffered_roads['LINK_ID'].tolist()
print(f'처리된 도로 구간: {len(all_link_ids):,}개')

In [ ]:
# ── Step 3. 불법주정차 Map Matching + 시계열 생성 ────────────────────────────
parking_matched = spatial_join_points_to_roads(
    parking_df, buffered_roads, lon_col='경도', lat_col='위도', time_col='단속일시')
pivot_df = build_timeseries_pivot(parking_matched, TRAIN_START, TEST_END, all_link_ids)
static_context, temporal_context, scaler = build_model_inputs(pivot_df, buffered_roads)
print(f'pivot_df: {pivot_df.shape}')

In [ ]:
# ── Step 4. SVI 파이프라인 로드/구성 ─────────────────────────────────────────

# 4-1. 캡션 로드 (기존 유지)
with open(CAPTION_OUTPUT, 'r', encoding='utf-8') as f:
    captions = json.load(f)
meta_df = pd.DataFrame([r for r in captions if r.get('caption')]
                       )[['location_id', 'LINK_ID', 'date']]
print(f'캡션 로드: {len(captions):,}개')

# 4-2. SBERT 임베딩 로드 (ablation study 비교용으로 보존)
#embeddings = np.load(EMBEDDING_OUTPUT)
#print(f'임베딩 로드: {embeddings.shape}')

# 4-3. 키워드 점수 로드 ──────────────────────────────── [수정: 이미 LINK_ID별 집계됨]
with open(KEYWORD_OUTPUT, 'r', encoding='utf-8') as f:
    kw_raw = json.load(f)

kw_df         = pd.DataFrame(kw_raw)
kw_df['LINK_ID'] = kw_df['LINK_ID'].astype(int)
KW_COLS       = [c for c in kw_df.columns if c != 'LINK_ID']  # 35개 키워드 자동 추출

# 전체 도로에 맞춰 정렬 (SVI 없는 도로는 0으로 채움)
kw_by_link    = (kw_df.set_index('LINK_ID')[KW_COLS]   # groupby 불필요 — 이미 집계됨
                 .reindex(all_link_ids)
                 .fillna(0.0))

keyword_context = kw_by_link.values.astype(np.float32)  # (N, 35)
KC = torch.FloatTensor(keyword_context).to(device)

n_with_svi = (kw_by_link.sum(axis=1) > 0).sum()
print(f'키워드 벡터: {keyword_context.shape}')
print(f'ℹ️  SVI 보유: {n_with_svi:,} / 전체: {len(all_link_ids):,}개')

# 4-4. 키워드 유사도 행렬 계산 ──────────────────────────────── [변경: SBERT → keyword]
KW_SIM_OUTPUT = 'sim_matrix_keyword.npy'

if Path(KW_SIM_OUTPUT).exists():
    print(f'📂 키워드 유사도 행렬 로드: {KW_SIM_OUTPUT}')
    sim_matrix_visual = np.load(KW_SIM_OUTPUT)
else:
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim
    print('🔢 키워드 유사도 행렬 계산 중...')
    sim_matrix_visual = cos_sim(keyword_context).astype(np.float32)
    np.fill_diagonal(sim_matrix_visual, 0.0)
    np.save(KW_SIM_OUTPUT, sim_matrix_visual)
    print(f'✅ 키워드 유사도 행렬: {sim_matrix_visual.shape} | 저장: {KW_SIM_OUTPUT}')

In [ ]:
if Path(ADJ_VISUAL_OUTPUT).exists():
    print(f'📂 adj_visual 로드: {ADJ_VISUAL_OUTPUT}')
    adj_visual_sparse = torch.load(ADJ_VISUAL_OUTPUT)
else:
    print(f'adj_visual 구성 중 (threshold={VISUAL_THRESHOLD})...')
    adj_visual_sparse = build_adj_threshold_topk(
        sim_matrix_visual, VISUAL_THRESHOLD, k=20, label='visual', device='cuda')
    torch.save(adj_visual_sparse, ADJ_VISUAL_OUTPUT)

In [ ]:
# ── Step 5. adj_phys + adj_adapt 구성 ────────────────────────────────────────

# Define the corrected build_adj_phys function locally to fix the error
adj_phys_dense, G = build_adj_phys(buffered_roads, pivot_df)

if Path(ADJ_ADAPT_OUTPUT).exists():
    print(f'📂 adj_adapt 로드: {ADJ_ADAPT_OUTPUT}')
    adj_adapt_sparse = torch.load(ADJ_ADAPT_OUTPUT)
else:
    sc      = torch.FloatTensor(static_context)
    sc_norm = F.normalize(sc, dim=-1)
    sim_adapt = (sc_norm @ sc_norm.T).numpy(); np.fill_diagonal(sim_adapt, 0)
    adj_adapt_sparse = build_adj_threshold_topk(sim_adapt, ADAPT_THRESHOLD, k=20, label='adapt', device='cuda')

    torch.save(adj_adapt_sparse, ADJ_ADAPT_OUTPUT)

In [ ]:
# ── Step 6. DataLoader + 모델 초기화 ─────────────────────────────────────────
train_loader, val_loader, test_loader = build_dataloaders(
    pivot_df, temporal_context,
    TRAIN_START, TRAIN_END, VAL_START, VAL_END, TEST_START, TEST_END)

D_static  = static_context.shape[1]
D_keyword = keyword_context.shape[1]                                    # [신규] 35
#model     = MCAST_ZINB(static_dim=D_static,
#                       keyword_dim=D_keyword).to(device)                # [수정]
#print(f'✅ 파라미터: {sum(p.numel() for p in model.parameters()):,}개')
#print(f'   static_dim={D_static} | keyword_dim={D_keyword}')

SC = torch.FloatTensor(static_context).to(device)
# KC는 Step 4에서 이미 생성됨
AP = adj_phys_dense.to(device)
AA = adj_adapt_sparse.to(device)
AV = adj_visual_sparse.to(device)

In [ ]:
# ── Step 7. 학습 ──────────────────────────────────────────────────────────────
history = run_training(
    model, train_loader, val_loader,
    SC, KC, AP, AA, AV,
    n_epochs=EPOCHS,        # 원래 값 70
    lr=LR, warmup_epochs=30, device=device,
    save_path='best_model_multistep.pt')

In [ ]:
# ── Step 8. 테스트 평가 ───────────────────────────────────────────────────────
model.load_state_dict(torch.load('best_model_multistep.pt'))
test_metrics = evaluate(model, test_loader,
                        SC, KC, AP, AA, AV, device=device)             # [KC 추가]
print_metrics(test_metrics, split='Test')

In [ ]:
per_h, overall = evaluate_multistep(model, test_loader, SC, KC, AP, AA, AV)

In [ ]:
# 전체 평균
print("=== 전체 평균 ===")
print(f"loss={overall['loss']:.4f}  MAE={overall['mae']:.4f}  RMSE={overall['rmse']:.4f}")
print(f"F1={overall['f1_pos']:.4f}  HR@20={overall['hr_at_20']:.4f}")

# horizon별 — 값이 서로 다른지가 핵심
print("\n=== horizon별 ===")
print(f"{'h':>3} {'MAE':>8} {'RMSE':>8} {'NZ-MAE':>8} {'F1':>8} {'HR@20':>8}")
for r in per_h:
    print(f"{r['h']:>3} {r['mae']:>8.4f} {r['rmse']:>8.4f} "
          f"{r['mae_pos']:>8.4f} {r['f1_pos']:>8.4f} {r['hr_at_20']:>8.4f}")

In [ ]:
import json
with open('/content/drive/MyDrive/mcast_data/mcast_multistep_result.json', 'w') as f:
    json.dump({'overall': overall, 'per_horizon': per_h}, f, ensure_ascii=False, indent=2)

In [ ]:
# ── Step 9. 예측 결과 저장 ────────────────────────────────────────────────────
@torch.no_grad()
def save_prediction_matrix(model, dataloader, pivot_df,
                             static_context, keyword_context,          # [keyword_context 추가]
                             adj_phys, adj_adapt, adj_visual,
                             output_path='predictions.csv', device=device):
    """테스트 세트 전체 예측 결과 저장 + 학습된 GCN 가중치(α, β, γ) 출력."""
    model.eval()
    print('📊 학습된 GCN 가중치:')
    for i, block in enumerate(model.blocks):
        w = F.softmax(block.gcn_weights, dim=0).detach().cpu().numpy()
        print(f'  Block {i}: α(phys)={w[0]:.3f} | β(adapt)={w[1]:.3f} | γ(visual)={w[2]:.3f}')

    all_times = pivot_df.columns
    valid_indices = dataloader.dataset.valid_indices

    all_time_stamps   = []
    all_nodes         = []
    all_pi_flat       = []
    all_mu_flat       = []
    all_theta_flat    = []
    all_expected_flat = []
    all_y_true_flat   = []
    sample_offset     = 0

    for x_r, x_d, x_w, temporal, target in dataloader:
        B = x_r.size(0)
        pi, mu, theta = model(x_r.to(device), x_d.to(device), x_w.to(device),
                               temporal.to(device), static_context,
                               keyword_context,                        # [추가]
                               adj_phys, adj_adapt, adj_visual)
        # 이하 동일
        expected = ((1 - pi) * mu).cpu().numpy()
        y_np     = target.numpy(); pi_np = pi.cpu().numpy()
        mu_np    = mu.cpu().numpy(); theta_np = theta.cpu().numpy()

        num_nodes_in_batch = mu.shape[1]
        batch_times = [all_times[valid_indices[sample_offset + i]] for i in range(B)]

        all_time_stamps.extend(np.repeat(batch_times, num_nodes_in_batch))
        all_nodes.extend(np.tile(np.arange(num_nodes_in_batch), B))
        all_pi_flat.extend(pi_np.flatten())
        all_mu_flat.extend(mu_np.flatten())
        all_theta_flat.extend(theta_np.flatten())
        all_expected_flat.extend(expected.flatten())
        all_y_true_flat.extend(y_np.flatten())
        sample_offset += B

    df = pd.DataFrame({
        'time'    : all_time_stamps,
        'node'    : all_nodes,
        'pi'      : all_pi_flat,
        'mu'      : all_mu_flat,
        'theta'   : all_theta_flat,
        'expected': all_expected_flat,
        'y_true'  : all_y_true_flat
    })

    df_wide = df.pivot(index='time', columns='node',
                       values=['pi', 'mu', 'theta', 'expected', 'y_true'])
    df_wide.columns = [f'{m}_node{n}' for m, n in df_wide.columns]
    df_wide.to_csv(output_path)
    print(f'✅ 예측 결과 저장: {output_path} | shape={df_wide.shape}')
    return df_wide

# ── Step 9. 예측 결과 저장 ────────────────────────────────────────────────────
df_preds = save_prediction_matrix(model, test_loader, pivot_df, SC, KC, AP, AA, AV)                 # [KC 추가]

# Google Drive 저장 (선택)
import shutil
shutil.copy('predictions.csv', f'{DRIVE_BASE}/predictions_MCAST_5.csv')

---
## SECTION 7-C Baseline

In [ ]:
# [공통 설정 셀] — 런타임 시작/재시작마다 실행
# baseline_models_multistep.py 는 저장소의 src/ 폴더에 있습니다.
exec(open('./src/baseline_models_multistep.py').read())          # 로컬 실행
# exec(open('/content/drive/MyDrive/<your_path>/src/baseline_models_multistep.py').read())  # Colab


In [ ]:
BL_CKPT_DIR     = '/content/drive/MyDrive/mcast_data/multistep_ckpt'  # ← 핵심
BL_RESULTS_PATH = '/content/drive/MyDrive/mcast_data/bl_multistep_results.json'

In [ ]:
import os
os.makedirs(BL_CKPT_DIR, exist_ok=True)
print('체크포인트 폴더 준비:', BL_CKPT_DIR)

In [ ]:
batch = next(iter(test_loader))
cost = {}
cost['MCAST-ZINB'] = measure_cost(model,
    build_forward_fn(model, batch, device=device, mcast_args=dict(
        static_context=SC, keyword_context=KC,
        adj_phys=AP, adj_adapt=AA, adj_visual=AV)), device)
cost_table(cost)

In [ ]:
print_progress()  # ✅ HA만 남아있는지 확인

In [ ]:
run_bl_ha(pivot_df, TRAIN_START, TRAIN_END, TEST_START, TEST_END,
          h_recent=H_RECENT, h_weekly=H_WEEKLY)

In [ ]:
run_bl_lstm(train_loader, val_loader, test_loader, device=device, hidden_dim=HIDDEN_DIM, epochs=EPOCHS, lr=LR)

In [ ]:
run_bl_gru(train_loader, val_loader, test_loader, device=device, hidden_dim=HIDDEN_DIM, epochs=EPOCHS, lr=LR)

In [ ]:
# OOM 시 BL_BATCH_SIZE_GNN = 4 (또는 2) 로 줄인 후 재실행
run_bl_stgcn(train_loader, val_loader, test_loader, AP, device=device, hidden_dim=HIDDEN_DIM, epochs=EPOCHS, lr=LR)

In [ ]:
run_bl_dcrnn(train_loader, val_loader, test_loader, AP,
             device=device, hidden_dim=HIDDEN_DIM, epochs=EPOCHS, lr=LR)

In [ ]:
run_bl_gwn(train_loader, val_loader, test_loader, AP,
           device=device, hidden_dim=HIDDEN_DIM, epochs=EPOCHS, lr=LR)

In [ ]:
run_bl_astgcn(train_loader, val_loader, test_loader, AP,
              device=device, hidden_dim=HIDDEN_DIM, epochs=EPOCHS, lr=LR, T_in=H_RECENT)

In [ ]:
results = load_bl_results()

# MCAST 결과를 같은 형식으로 합치기 (본 노트북에서)
per_h, overall = evaluate_multistep(model, test_loader, SC, KC, AP, AA, AV)
results['MCAST-ZINB'] = {**overall, 'per_horizon': per_h}

order = ['HA','LSTM','GRU','STGCN','DCRNN','GraphWaveNet','ASTGCN', 'MCAST-ZINB']
df = report_wide(results, horizons=(1, 3, 6), model_order=order)
print(df.to_string())
save_report_wide(df, 'horizon_report')

---
## SECTION 7-D: Ablation

In [ ]:
import sys
# baseline / ablation 모듈은 저장소의 src/ 폴더에 있습니다.
sys.path.append('./src')                                    # 로컬 실행
# sys.path.append('/content/drive/MyDrive/<your_path>/src') # Colab: Drive에 올린 경우
from ablation_multistep import run_ablation_study


In [ ]:
results = run_ablation_study(
    train_loader, val_loader, test_loader,
    SC, KC, AP, AA, AV,
    D_static      = static_context.shape[1],
    D_keyword     = keyword_context.shape[1],
    N             = len(all_link_ids),
    epochs        = EPOCHS,
    lr            = LR,
    warmup_epochs = 30,
    device        = device,
    save_dir      = '/content/drive/MyDrive/mcast_data/ablation/multistep/',  # ← Drive 경로
)

---
## SECTION 8-1 : 시각화 (2)

In [ ]:
# ================================================================
# 시각화
# ================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd # Import pandas here since it's used in read_csv and DatetimeIndex operations

def plot_daily_total(df_wide, date_str, label='MCAST-ZINB',
                     figsize=(14, 5), save_path=None):
    """
    특정 날짜의 시간대별 전체 노드 예측값 합 vs 실제값 합 비교.

    Args:
        df_wide   : save_prediction_matrix() 반환값 (또는 CSV 로드)
        date_str  : 'YYYY-MM-DD' 형식
        label     : 예측값 범례 이름
        save_path : None이면 화면 출력, 경로 지정 시 파일 저장
    """
    mask     = df_wide.index.to_series().dt.date == pd.to_datetime(date_str).date()
    df_day   = df_wide[mask]

    if len(df_day) == 0:
        print(f'⚠️  {date_str} 데이터 없음')
        return

    sum_pred = round(df_day.filter(regex='expected_node')).sum(axis=1)
    sum_true = round(df_day.filter(regex='y_true_node')).sum(axis=1)
    hours    = df_day.index.hour

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(hours, sum_pred.values, marker='x', linewidth=2,
            label=f'Predicted ({label})')
    ax.plot(hours, sum_true.values, marker='o', linewidth=2,
            color='brown', label='Ground Truth')

    ax.set_xlabel('Hour of Day', fontsize=12)
    ax.set_ylabel('Total Count (all nodes)', fontsize=12)
    ax.set_title(f'Predicted vs Ground Truth — {date_str}', fontsize=13)
    ax.set_xticks(range(24))
    ax.grid(True, alpha=0.4)
    ax.legend(fontsize=11)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'✅ 저장: {save_path}')
    else:
        plt.show()


def plot_node_timeseries(df_wide, node_id, start_date=None, end_date=None,
                         label='MCAST-ZINB', figsize=(16, 4), save_path=None):
    """
    특정 노드의 전체 기간(또는 지정 기간) 예측값 vs 실제값 시계열 비교.

    Args:
        df_wide    : save_prediction_matrix() 반환값
        node_id    : 노드 번호 (int)
        start_date : 'YYYY-MM-DD' 또는 None (전체)
        end_date   : 'YYYY-MM-DD' 또는 None (전체)
        label      : 예측값 범례 이름
    """
    pred_col = f'expected_node{node_id}'
    true_col = f'y_true_node{node_id}'

    if pred_col not in df_wide.columns:
        print(f'⚠️  node {node_id} 없음')
        return

    df = df_wide[[pred_col, true_col]].copy()
    if start_date:
        df = df[df.index >= start_date]
    if end_date:
        df = df[df.index <= end_date]

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df.index, df[true_col].values,  color='brown',
            linewidth=1.2, label='Ground Truth', alpha=0.8)
    ax.plot(df.index, df[pred_col].values,  color='steelblue',
            linewidth=1.2, label=f'Predicted ({label})', alpha=0.8)

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'Node {node_id} — Predicted vs Ground Truth', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
    else:
        plt.show()


def plot_zero_rate_over_time(df_wide, freq='D', label='MCAST-ZINB',
                              figsize=(14, 4), save_path=None):
    """
    시간에 따른 실제/예측 zero 비율 추이 비교.
    freq: 'D'=일별, 'W'=주별, 'H'=시간별
    """
    pred_cols = df_wide.filter(regex='expected_node')
    true_cols = df_wide.filter(regex='y_true_node')

    pred_zero = (pred_cols.round() == 0).mean(axis=1).resample(freq).mean()
    true_zero = (true_cols == 0).mean(axis=1).resample(freq).mean()

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(true_zero.index, true_zero.values, color='brown',
            linewidth=1.5, label='True Zero Rate')
    ax.plot(pred_zero.index, pred_zero.values, color='steelblue',
            linewidth=1.5, linestyle='--', label=f'Pred Zero Rate ({label})')

    ax.set_ylabel('Zero Rate', fontsize=12)
    ax.set_title('Zero Rate: Ground Truth vs Predicted', fontsize=13)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
    else:
        plt.show()


# ── CSV 로드 후 시각화 ─────────────────────────────────────────────
df_preds = pd.read_csv(
    'predictions_MCAST_4.csv',
    index_col='time',
    parse_dates=True)

# Explicitly convert index to DatetimeIndex to ensure .dt accessor works
df_preds.index = pd.to_datetime(df_preds.index, format='mixed')

# 1) 특정 날짜 시간대별 합계 비교
plot_daily_total(df_preds, date_str='2024-01-13', label='MCAST-ZINB')

# 2) 특정 노드 시계열 비교 (예: 노드 42번)
plot_node_timeseries(df_preds, node_id=42)

# 3) Zero rate 추이 비교
plot_zero_rate_over_time(df_preds, freq='D')

In [ ]:
# 1) 특정 날짜 시간대별 합계 비교
plot_daily_total(df_preds, date_str='2024-01-23', label='MCAST-ZINB')

In [ ]:
# 1) 특정 날짜 시간대별 합계 비교
plot_daily_total(df_preds, date_str='2024-01-24', label='MCAST-ZINB')

In [ ]:
# 2) 특정 노드 시계열 비교 (예: 노드 42번)
plot_node_timeseries(df_preds, node_id=5038)

In [ ]:
# 2) 특정 노드 시계열 비교 (예: 노드 42번)
plot_node_timeseries(df_preds, node_id=5040)

In [ ]:
# 2) 특정 노드 시계열 비교 (예: 노드 42번)
plot_node_timeseries(df_preds, node_id=5042)

In [ ]:
# 2) 특정 노드 시계열 비교 (예: 노드 42번)
plot_node_timeseries(df_preds, node_id=5043)

In [ ]:
# 1) 특정 날짜 시간대별 합계 비교
plot_daily_total(df_preds, date_str='2024-01-27', label='MCAST-ZINB')

In [ ]:
# 1) 특정 날짜 시간대별 합계 비교
plot_daily_total(df_preds, date_str='2024-02-24', label='MCAST-ZINB')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_total_timeseries(df_wide, start_date=None, end_date=None,
                         label='CST-ZINB', figsize=(16, 6), save_path=None):
    """
    지정 기간 동안 전체 노드의 총 예측값 vs 실제값 시계열 비교.

    Args:
        df_wide    : save_prediction_matrix() 반환값
        start_date : 'YYYY-MM-DD' 또는 None (전체)
        end_date   : 'YYYY-MM-DD' 또는 None (전체)
        label      : 예측값 범례 이름
    """
    df = df_wide.copy()
    if start_date:
        df = df[df.index >= start_date]
    if end_date:
        df = df[df.index <= end_date]

    if len(df) == 0:
        print(f'⚠️  {start_date} ~ {end_date} 기간에 데이터가 없습니다.')
        return

    sum_pred = (df.filter(regex='expected_node')).round().sum(axis=1)
    sum_true = df.filter(regex='y_true_node').sum(axis=1)

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df.index, sum_true.values,  color='brown',
            linewidth=1.5, label='Ground Truth', alpha=0.8)
    ax.plot(df.index, sum_pred.values,  color='steelblue',
            linewidth=1.5, label=f'Predicted ({label})', alpha=0.8)

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1)) # Show ticks for each day
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Total Count (all nodes)', fontsize=12)
    ax.set_title(f'Total Predicted vs Ground Truth ({start_date} to {end_date})', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'✅ 저장: {save_path}')
    else:
        plt.show()

plot_total_timeseries(
    df_preds,
    start_date='2024-02-15',
    end_date='2024-02-29',
    label='MCAST-ZINB'
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_total_timeseries(df_wide, start_date=None, end_date=None,
                         label='CST-ZINB', figsize=(16, 6), save_path=None):
    """
    지정 기간 동안 전체 노드의 총 예측값 vs 실제값 시계열 비교.

    Args:
        df_wide    : save_prediction_matrix() 반환값
        start_date : 'YYYY-MM-DD' 또는 None (전체)
        end_date   : 'YYYY-MM-DD' 또는 None (전체)
        label      : 예측값 범례 이름
    """
    df = df_wide.copy()
    if start_date:
        df = df[df.index >= start_date]
    if end_date:
        df = df[df.index <= end_date]

    if len(df) == 0:
        print(f'⚠️  {start_date} ~ {end_date} 기간에 데이터가 없습니다.')
        return

    sum_pred = (df.filter(regex='expected_node')).round().sum(axis=1)
    sum_true = df.filter(regex='y_true_node').sum(axis=1)

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df.index, sum_true.values,  color='brown',
            linewidth=1.5, label='Ground Truth', alpha=0.8)
    ax.plot(df.index, sum_pred.values,  color='steelblue',
            linewidth=1.5, label=f'Predicted ({label})', alpha=0.8)

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1)) # Show ticks for each day
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Total Count (all nodes)', fontsize=12)
    ax.set_title(f'Total Predicted vs Ground Truth ({start_date} to {end_date})', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'✅ 저장: {save_path}')
    else:
        plt.show()

plot_total_timeseries(
    df_preds,
    start_date='2024-01-16',
    end_date='2024-02-01',
    label='MCAST-ZINB'
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_total_timeseries(df_wide, start_date=None, end_date=None,
                         label='CST-ZINB', figsize=(16, 6), save_path=None):
    """
    지정 기간 동안 전체 노드의 총 예측값 vs 실제값 시계열 비교.

    Args:
        df_wide    : save_prediction_matrix() 반환값
        start_date : 'YYYY-MM-DD' 또는 None (전체)
        end_date   : 'YYYY-MM-DD' 또는 None (전체)
        label      : 예측값 범례 이름
    """
    df = df_wide.copy()
    if start_date:
        df = df[df.index >= start_date]
    if end_date:
        df = df[df.index <= end_date]

    if len(df) == 0:
        print(f'⚠️  {start_date} ~ {end_date} 기간에 데이터가 없습니다.')
        return

    sum_pred = (df.filter(regex='expected_node')).round().sum(axis=1)
    sum_true = df.filter(regex='y_true_node').sum(axis=1)

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df.index, sum_true.values,  color='brown',
            linewidth=1.5, label='Ground Truth', alpha=0.8)
    ax.plot(df.index, sum_pred.values,  color='steelblue',
            linewidth=1.5, label=f'Predicted ({label})', alpha=0.8)

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1)) # Show ticks for each day
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Total Count (all nodes)', fontsize=12)
    ax.set_title(f'Total Predicted vs Ground Truth ({start_date} to {end_date})', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'✅ 저장: {save_path}')
    else:
        plt.show()

plot_total_timeseries(
    df_preds,
    start_date='2024-02-15',
    end_date='2024-02-29',
    label='MCAST-ZINB'
)

In [ ]:
# ── R1-W4 대응: SVI 보유 vs 비보유 구간 성능 분해 (재학습·재추론 없음) ──────────────
import pandas as pd, numpy as np, re
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1) 저장된 예측 결과 로드 (predictions.csv 만 사용)
df_preds = pd.read_csv('predictions_MCAST_4.csv', index_col=0, parse_dates=True)

# 2) expected / y_true 를 노드 인덱스(0..N-1) 순서로 정렬해 (T x N) 행렬로 복원
#    filter regex는 문자열 정렬이라 node10<node2 문제 발생 → 정수로 재정렬 필수
def wide_to_matrix(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix)]
    ids  = np.array([int(re.search(r'node(\d+)$', c).group(1)) for c in cols])
    order = np.argsort(ids)
    return df[[cols[i] for i in order]].to_numpy(), ids[order]

expected, nid = wide_to_matrix(df_preds, 'expected_node')   # (T, N)
y_true,   _   = wide_to_matrix(df_preds, 'y_true_node')      # (T, N)
T, N = expected.shape
assert (nid == np.arange(N)).all(), '노드 인덱스가 0..N-1 연속이 아님 — 매핑 확인'

# 3) 커버리지 마스크: KC(keyword_context, N×35)에서 도출. 비보유=zero-vector → 행합 0
KC_np = KC.detach().cpu().numpy() if hasattr(KC, 'detach') else np.asarray(KC)
covered = (KC_np != 0).any(axis=1)                          # (N,) True=SVI 보유
print(f'SVI 보유 {covered.sum()} / {N}  (비보유 {(~covered).sum()})')
# → 4579 / 5661 근처인지 확인. 다르면 all-zero 캡션 구간 때문이니
#   svi_keywords_by_link.json 키 목록으로 covered 를 직접 지정.

# 4) 포인트 단위 지표 (열 서브셋) — evaluate()와 동일 정의
def pointwise(exp, yt):
    yf, ef = yt.flatten(), exp.flatten()
    pos = yf > 0
    pred_bin, true_bin = (ef > 0.5).astype(int), (yf > 0).astype(int)
    tp = ((pred_bin==1)&(true_bin==1)).sum()
    fp = ((pred_bin==1)&(true_bin==0)).sum()
    fn = ((pred_bin==0)&(true_bin==1)).sum()
    prec, rec = tp/(tp+fp+1e-8), tp/(tp+fn+1e-8)
    return dict(
        MAE=mean_absolute_error(yf, ef),
        RMSE=np.sqrt(mean_squared_error(yf, ef)),
        NonzeroMAE=(mean_absolute_error(yf[pos], ef[pos]) if pos.sum() else np.nan),
        F1=2*prec*rec/(prec+rec+1e-8),
        pos_samples=int(pos.sum()))

# 5) HR@20% — 전역 랭킹 유지 분해 (논문 evaluate()의 top-k 로직 그대로 재현)
def hr20_decomp(exp, yt, cov):
    T, N = exp.shape
    k = max(1, int(np.floor(0.2 * N)))          # 전체 N의 20% (논문과 동일)
    hits = dict(cov=0, non=0); events = dict(cov=0, non=0)
    for t in range(T):
        true_nodes = np.where(yt[t] > 0)[0]
        if true_nodes.size == 0: continue
        topk = np.argsort(exp[t])[::-1][:k]      # 전체 N 기준 상위 k (원본과 동일)
        mask_top = np.zeros(N, bool); mask_top[topk] = True
        hit_nodes = true_nodes[mask_top[true_nodes]]
        tc = cov[true_nodes]; hc = cov[hit_nodes]
        events['cov'] += int(tc.sum());  events['non'] += int((~tc).sum())
        hits['cov']   += int(hc.sum());  hits['non']   += int((~hc).sum())
    hr = lambda h,e: (h/e if e else np.nan)
    return {
        'covered':     hr(hits['cov'], events['cov']),
        'non_covered': hr(hits['non'], events['non']),
        'overall':     hr(hits['cov']+hits['non'], events['cov']+events['non']),
        'events_cov': events['cov'], 'events_non': events['non']}

# 6) 표 조립
rows = {}
for name, m in [('SVI 보유', covered), ('SVI 비보유', ~covered), ('전체', np.ones(N, bool))]:
    pm = pointwise(expected[:, m], y_true[:, m])
    rows[name] = pm
hr = hr20_decomp(expected, y_true, covered)

res = pd.DataFrame(rows).T[['NonzeroMAE','F1','MAE','RMSE','pos_samples']]
res['HR@20%'] = [hr['covered'], hr['non_covered'], hr['overall']]
print('\n', res.round(4).to_string())
print(f"\nHR@20% 이벤트 수 — 보유:{hr['events_cov']}  비보유:{hr['events_non']}")
print(f"overall HR@20% = {hr['overall']:.4f}  (논문 0.7244 와 일치하는지 확인)")